# Building conversational applications with Strands Agents

In this notebook, you learn how to use Strands agents to integrate external capabilities into conversational applications.

This is a conversion of the original module9-converse_with_tools.ipynb that used the Converse API with tool specifications. Instead, we'll use Strands agents for a more streamlined approach.

## Environment setup

In this task, you set up your environment and install the required packages for Strands agents.

In [4]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.tools.mcp import MCPClient

model_id = "amazon.nova-lite-v1:0"

## Define the MCP Server command

In [5]:
# Define the MCP server using stdio transport
stdio_mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="uv", 
        args=["run", "module9-mcp_server.py"]
    )
))

## Instantiate the MCP Client and create the Strands Agent

Now we create a Strands agent that uses the Bedrock LLM and our defined tools. The agent will automatically handle tool calling and conversation flow.

In [6]:
import sys
from pathlib import Path

from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

model_id = "amazon.nova-lite-v1:0"

server_file = Path.cwd() / "module9-mcp_server.py"

if not server_file.exists():
    raise FileNotFoundError(f"Cannot find MCP server file: {server_file}")

mcp_errlog = open("mcp-server-error.log", "w", encoding="utf-8")

stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[str(server_file)],
            cwd=str(Path.cwd())
        ),
        errlog=mcp_errlog
    )
)

In [7]:
# Create the Bedrock LLM wrapper for Strands
llm = BedrockModel(
    model_id=model_id
)

# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model=llm,
        tools=tools,
        system_prompt="You are a helpful assistant that can perform calculations and search for information on DuckDuckGo."
    )

    # Send the question
    agent("What is Amazon SageMaker? What is launch year multiplied by 2")

<thinking>The User has asked two questions: one about the definition of Amazon SageMaker and another about the launch year of Amazon SageMaker multiplied by 2. To answer the first question, I need to search for information about Amazon SageMaker. To answer the second question, I need to know the launch year of Amazon SageMaker and then perform a multiplication operation.</thinking>

Tool #1: duckduckgo_search
<thinking>I have found information about Amazon SageMaker from the search result. Now, I need to find out the launch year of Amazon SageMaker to multiply it by 2. I will use another tool to search for this information.</thinking> 
Tool #2: duckduckgo_search
<thinking>I have found that Amazon SageMaker was launched in 2017. Now, I will calculate the launch year multiplied by 2.</thinking> 
Tool #3: calculator
<thinking>I have completed both parts of the User's request. I will now provide the answers to the User.</thinking>

Amazon SageMaker is a fully managed machine learning platf